# M4 Practice: Data Wrangling with NTL-LTER Data

This practice notebook provides additional preparation for Assignment 4. You will use the North Temperate Lakes Long-Term Ecological Research (NTL-LTER) lake chemistry and physics dataset to practice selecting, transforming, combining, summarizing, and reshaping data with pandas.

The exercises are intentionally open-ended. Complete each code cell and inspect your results before moving on. Use the techniques from both M4 lesson notebooks.

## Learning goals

By the end of this practice, you should be able to:

- import a raw environmental dataset and inspect its structure;
- filter rows with Boolean masks, `.query()`, and `.isin()`;
- identify and handle missing values;
- select and rename columns;
- create variables from dates and existing measurements;
- combine compatible dataframes with `pd.concat()`;
- create grouped summaries; and
- reshape data between long and wide formats.

## 1. Set up the session and inspect the raw data

In [1]:
# 1.1 Import packages
from pathlib import Path
import pandas as pd

In [2]:
# 1.2 Set folder paths (project, raw, processed)
project_fldr = Path.cwd().parent
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
raw_fldr

WindowsPath('c:/Workspace/Teaching_2026Fall/EDE_Python_development/data/raw')

In [3]:
# 1.3. Read the raw data in as a dataframe
NTL = pd.read_csv(
    raw_fldr / 'NTL-LTER_Lake_ChemistryPhysics_Raw.csv',
    dtype={'lakeid': 'category', 'lakename': 'category'},
    parse_dates=['sampledate'],
    date_format='%m/%d/%y'
)
NTL.head()

,lakeid,lakename,year4,daynum,sampledate,depth,temperature_C,dissolvedOxygen,irradianceWater,irradianceDeck,comments
0,L,Paul Lake,1984,148,1984-05-27,0.00,14.5,9.5,1750.0,1620.0,NaN
1,L,Paul Lake,1984,148,1984-05-27,0.25,NaN,NaN,1550.0,1620.0,NaN
2,L,Paul Lake,1984,148,1984-05-27,0.50,NaN,NaN,1150.0,1620.0,NaN
3,L,Paul Lake,1984,148,1984-05-27,0.75,NaN,NaN,975.0,1620.0,NaN
4,L,Paul Lake,1984,148,1984-05-27,1.00,14.5,8.8,870.0,1620.0,NaN


### Exercise 1

Use `.shape`, `.info()`, and `.describe()` to inspect the dataset. Then answer in a markdown cell:

1. How many rows and columns are present?
2. Which columns contain dates, categories, and measurements?
3. What is one feature of the dataset that may require wrangling before analysis?
4. Why isn't the columns column converted into a categorical column?

In [ ]:
# Ex1.1 Reveal the dimensions of the dataframe


(38614, 11)

In [ ]:
# Ex1.2 Reveal column information on the dataframe


<class 'pandas.DataFrame'>
RangeIndex: 38614 entries, 0 to 38613
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   lakeid           38614 non-null  category      
 1   lakename         38614 non-null  category      
 2   year4            38614 non-null  int64         
 3   daynum           38614 non-null  int64         
 4   sampledate       38614 non-null  datetime64[us]
 5   depth            38614 non-null  float64       
 6   temperature_C    34756 non-null  float64       
 7   dissolvedOxygen  34575 non-null  float64       
 8   irradianceWater  24327 non-null  float64       
 9   irradianceDeck   23195 non-null  float64       
 10  comments         368 non-null    str           
dtypes: category(2), datetime64[us](1), float64(5), int64(2), str(1)
memory usage: 2.7 MB


In [ ]:
# Ex1.3 Reveal summary statistics for numeric columns


,year4,daynum,sampledate,depth,temperature_C,dissolvedOxygen,irradianceWater,irradianceDeck
count,38614.000000,38614.00000,38614,38614.000000,34756.000000,34575.000000,24327.000000,23195.000000
mean,1998.567152,194.26941,1999-02-05 06:57:09.015383,4.390269,11.808707,4.970496,210.241580,720.508137
min,1984.000000,55.00000,1984-05-27 00:00:00,0.000000,0.300000,0.000000,-0.337000,1.500000
25%,1991.000000,166.00000,1991-08-08 00:00:00,1.500000,5.300000,0.300000,14.000000,353.000000
50%,1997.000000,194.00000,1997-07-28 00:00:00,4.000000,9.300000,5.600000,65.000000,747.000000
75%,2006.000000,222.00000,2006-06-06 00:00:00,6.500000,18.700000,8.400000,265.000000,1042.000000
max,2016.000000,307.00000,2016-08-17 00:00:00,20.000000,34.100000,802.000000,24108.000000,8532.000000
std,9.065429,33.52642,NaN,3.413634,6.939101,8.724565,345.452012,409.229792


## 2. Filter and select an analysis subset

Create a subset for three lakes: Peter Lake, Paul Lake, and Tuesday Lake. Keep only surface observations (`depth == 0`) collected from 2000 onward. Use `.loc[]` and `.isin()` for the filter.

Name the result `surface_three_lakes`. Report its dimensions and the number of observations contributed by each lake.
>Note: You can remove unused lakename categories with `surface_three_lakes['lakename'].cat.remove_unused_categories()`

In [ ]:
# 2.1 Filter selected lakes into a new dataframe
# Create a list of the lakes to select
selected_lakes = ['Peter Lake', 'Paul Lake', 'Tuesday Lake']

# Code to create surface_three_lakes here


In [ ]:
# 2.2 Report dimensions and counts by lake


(592, 11)

In [ ]:
# 2.3 Report the number of records for each lake 


lakename
Peter Lake      249
Paul Lake       243
Tuesday Lake    100
Name: count, dtype: int64

### Exercise 2: Missing values

Using `surface_three_lakes`, determine:

1. How many missing values occur in each measurement column?
2. What percentage of irradiance (water) values are missing?
3. Create `complete_surface_three_lakes`, retaining only rows with non-missing temperature and dissolved oxygen.

Report the number of rows removed.

In [ ]:
# Ex2.1 Reveal the total number of missing values in each column


lakeid               0
lakename             0
year4                0
daynum               0
sampledate           0
depth                0
temperature_C        2
dissolvedOxygen      1
irradianceWater     27
irradianceDeck      27
comments           572
dtype: int64

In [ ]:
# Ex2.2 Report percentage of irradiance (water) values missing 


4.56% of irradiance (water) records are missing


In [ ]:
# Ex2.3 Extract complete records into new dataframe & report dimensions


2 were dropped for missing temperature or DO values


## 3. Select, rename, and create variables

Starting with `complete_surface_three_lakes`, keep only the lake name, sample date, depth, temperature, and dissolved oxygen columns. Rename them using snake_case names.

Then create:

- `year` and `month` from `sample_date`;
- `season` using the month-to-season dictionary from M4-2; and
- `temperature_F`, converting Celsius to Fahrenheit with `temperature_C * 9 / 5 + 32`.

Name the result `lake_features`.

In [13]:
#3.1 Create a season dictionary
season_dict = {
    12:'winter',1:'winter',2:'winter',
    3:'spring',4:'spring',5:'spring',
    6:'summer',7:'summer',8:'summer',
    9:'fall',10:'fall',11:'fall'
}

In [ ]:
# 3.2 Code wrangle complete_surface_three_lakes dataframe


In [ ]:
# 3.3 Preview the transformed data


,lake_name,sample_date,depth_m,temperature_C,dissolved_oxygen_mgL,season,temperature_F
23177,Paul Lake,2000-05-24,0.0,16.7,10.1,spring,62.06
23196,Peter Lake,2000-05-24,0.0,16.8,11.5,spring,62.24
23277,Tuesday Lake,2000-05-26,0.0,16.4,9.2,spring,61.52
23299,Paul Lake,2000-06-26,0.0,20.3,8.7,summer,68.54
23321,Peter Lake,2000-06-26,0.0,20.2,8.8,summer,68.36


## 4. Practice `pd.concat()`

`pd.concat()` stacks compatible dataframes. Here, create two dataframes from `lake_features`:

- `early_lakes`: observations from 2000 through 2009;
- `late_lakes`: observations from 2010 onward.

Use `pd.concat()` to recombine them into `recombined_lakes`. Set `ignore_index=True`. Compare the dimensions and lake counts of the original and recombined dataframes.

Explain in a markdown cell why `concat()` is appropriate here and why `merge()` would not be the primary operation.

In [ ]:
# 4.1 Create early_lakes and late_lakes dataframes


(299, 7) (291, 7)


In [ ]:
# 4.2 Recombine the dataframes with pd.concat()


(590, 7)

## 5. Grouped summaries

Using `recombined_lakes`, create a dataframe named `season_summary` with one row for each lake and season. Include:

- the number of records;
- mean temperature in Celsius;
- mean temperature in Fahrenheit; and
- mean dissolved oxygen.

Reset the index and sort the result by lake name and season.

In [ ]:
# 5.1 Group, summarize, and sort combined lake data


In [19]:
# 5.2 Examine results
season_summary

,lake_name,season,n_records,mean_temp_C,mean_temp_F,mean_do_mgL
0,Paul Lake,fall,2,18.900000,66.020000,7.800000
1,Paul Lake,spring,32,16.656250,61.981250,9.030625
2,Paul Lake,summer,209,22.028230,71.650813,7.625837
3,Peter Lake,fall,6,16.533333,61.760000,9.966667
4,Peter Lake,spring,33,16.645455,61.961818,9.608788
5,Peter Lake,summer,208,22.229327,72.012788,8.623221
6,Tuesday Lake,fall,5,15.140000,59.252000,8.380000
7,Tuesday Lake,spring,13,17.207692,62.973846,9.030769
8,Tuesday Lake,summer,82,22.112195,71.801951,8.005000


## 6. Reshape a summary from long to wide

Use `.pivot_table()` to create `season_temperature_wide`, with one row per lake and one column per season. The values should be mean temperature in Celsius.

Then use `.melt()` to convert it back to a long dataframe named `season_temperature_long`. Keep `lake_name` as an identifier and name the new columns `season` and `mean_temperature`.

In [ ]:
# 6.1 Create season_temperature_wide


In [ ]:
# 6.2 Create season_temperature_long


## 7. Capstone practice workflow

Create `practice_summary` in one readable workflow. It should:

1. start with `NTL`;
1. filter to Peter Lake, Paul Lake, and Tuesday Lake;
1. keep surface observations with non-missing temperature and dissolved oxygen;
1. create `year`, `month`, and `season`;
1. group by lake and year;
1. calculate record count, mean temperature, and mean dissolved oxygen;
1. reset the index; and
1. sort by lake and year.

Finally, save it as `NTL_practice_summary.csv` in the processed-data folder.

In [ ]:
# Build practice_summary here


,lakename,year,record_count,mean_temp,mean_DO
0,Paul Lake,1984,14,20.892857,7.057143
1,Paul Lake,1985,16,18.487500,7.475000
2,Paul Lake,1986,15,20.486667,7.346667
3,Paul Lake,1987,36,21.100000,7.147222
4,Paul Lake,1988,20,20.760000,7.365000


In [ ]:
# Export the processed summary here


## Reflection

In a markdown cell, briefly answer:

1. Which step required the most judgment?
2. How did filtering before summarizing affect the result?
3. What would you check before trusting a processed dataset created from several raw files?